In [3]:
# Instalações para Google Colab
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# Imports
import os, re, time, requests
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Função para limpar texto
def limpar(texto):
    return ' '.join(texto.strip().split()) if texto else "N/A"

# Data e pastas
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_casa_rossi"
os.makedirs(pasta_base, exist_ok=True)

# WebDriver Headless
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Coletar links
def coletar_links_produtos():
    url_base = 'https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado'
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url_base, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    ultima_pagina_tag = soup.select_one("li.page.last a")
    total_paginas = int(re.search(r'pg=(\d+)', ultima_pagina_tag['href']).group(1)) if ultima_pagina_tag else 1

    links = []
    for pagina in range(1, total_paginas + 1):
        url_pagina = f"{url_base}?pg={pagina}"
        response = requests.get(url_pagina, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        produtos = soup.find_all('li', class_='item flex')

        for produto in produtos:
            a_tag = produto.find('a', class_='space-image')
            if a_tag:
                link = a_tag['href']
                if not link.startswith("http"):
                    link = "https://www.casarossi.com.br" + link
                links.append(link)
        time.sleep(1)
        if len(links) >= 5:
            break
    return links[:5]

# Coleta dos dados
links_produtos = coletar_links_produtos()
dados = []

for i, url in enumerate(links_produtos):
    print(f"[{i+1}/5] Coletando: {url}")
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        nome = limpar(soup.find('h1', class_='product-name').text)
        codigo = limpar(soup.select_one('.product-sku .ref').text) if soup.select_one('.product-sku .ref') else nome[:20]
        preco_original = soup.select_one('.PrecoPrincipal span')
        preco_caixa = limpar(preco_original.text) if preco_original else "N/A"
        preco_m2 = soup.select_one('.preco-por-metro')
        preco_m2 = limpar(preco_m2.text.replace('por m²', '').replace('R$', '').strip()) if preco_m2 else "N/A"
        m2_caixa = soup.select_one('.metro-na-caixa')
        m2_caixa = limpar(m2_caixa.text.replace('Caixa com', '').replace('m²', '').strip()) if m2_caixa else "N/A"

        # Imagem
        imagem_tag = soup.select_one('.zoom img')
        imagem_url = imagem_tag['src'] if imagem_tag else "N/A"

        # DESCRIÇÃO: Agora pegamos o conteudo da div com id="descricao"
        descricao_html = soup.find('div', id='descricao')
        descricao = limpar(descricao_html.get_text(separator=' ')) if descricao_html else "N/A"

        # Especificações: extraindo os itens da div com id="caracteristicas"
        espec_div = soup.find('div', id='caracteristicas')
        if espec_div:
            itens = espec_div.find_all('li')
            especificacoes = ' | '.join([limpar(li.text) for li in itens])
        else:
            especificacoes = "N/A"

        # Comparar preços
        preco_promocional = preco_caixa
        if preco_m2 != "N/A" and preco_caixa != "N/A" and preco_caixa == preco_caixa:
            preco_promocional = "N/A"

        dados.append([
            codigo,
            nome,
            preco_promocional,
            preco_caixa,
            preco_m2,
            especificacoes,
            descricao,
            imagem_url,
            url,
            "Casa Rossi"
        ])

    except Exception as e:
        print(f"Erro no produto: {e}")

driver.quit()

# Criar DataFrame
df = pd.DataFrame(dados, columns=[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"
])

# Ajustar valores numéricos
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)
df["Preço Original"] = df["Preço Original"].apply(limpar_preco)
df["Preço Promocional"] = df.apply(
    lambda row: row["Preço Promocional"] if row["Preço Promocional"] != row["Preço Original"] else "N/A", axis=1
)

# Salvar CSV e Excel
csv_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.csv")
xlsx_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.xlsx")

df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
df.to_excel(xlsx_path, index=False)

print(f"✅ Arquivos salvos:\n- CSV: {csv_path}\n- Excel: {xlsx_path}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
[1/5] Coletando: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/piso-formigres-66x66cm-acetinado-retificado-coimbra-2-18m
[2/5] Coletando: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-retificado-delta-72x72cm-out-itaara-raven-2-61m
[3/5] Coletando: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanatoincepa-90x90cm-acetinado-duo-concret-cris-2-40m
[4/5] Coletando: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-helena-62x121cm-acetinado-120133-mint-2-25m
[5/5] Coletando: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-retificado-biancog